# Tracks with the highest local rain volume: accumulation maps and key frames

Reads the output of `runscripts/run_trackpcp.py --steps 2` (pixel table `trackpcp_pixels_*.parquet` and per-track metrics `trackpcp_metrics_*.parquet`), ranks the
tracks by a **local** rainfall metric (the largest mean depth in a box of about 45 x 45 km over the whole track: high rain volume in a small area, the flood-relevant case),
and plots for each of the top tracks

1. the accumulated precipitation over the track lifetime (`cmaps.WhBlGrYeRe`), title with start / end time and duration,
2. composite reflectivity at four key frames (start, peak area, peak hourly rain volume, end) with the track mask outlined.

Metrics (columns of the metrics table): `accum_max_mm` (largest pixel accumulation), `accum_vol_1e6m3` (water volume under the track), `local_depth_mm_<L>km` (largest box-mean depth in an
L x L km box), `local_rate_mm_h_<L>km` (largest box-mean *hourly* rain in that box: short-duration intensity), `area_ge<T>mm_km2`, `cluster_area_km2` / `cluster_vol_1e6m3`
(largest connected region of at least the cluster threshold). Rain is attributed to one track per pixel and hour (best class, then most steps).

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
import xarray as xr
import pyarrow.parquet as pq
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import colormaps as cmaps

DATA_DIR = "/global/cfs/cdirs/m1657/zfeng/MRMS/cell_mcs_conus/trackpcp/"      # output of run_trackpcp.py (steps 1 and 2)
STATS_DIR = "/pscratch/sd/i/iclas2/MRMS/mcs_conus_2025/stats/"
PIXEL_ROOT = "/pscratch/sd/i/iclas2/MRMS/mcs_conus_2025/mcstracking/"
STATS_FILEBASE = "mcs_tracks_final_"
FIG_DIR = DATA_DIR + "figures/"
os.makedirs(FIG_DIR, exist_ok=True)
RANK_BY = "local_depth_mm_45km"        # any column of the metrics table
NTOP = 5
CLASS_VALUES = None                    # e.g. [1] for MCS only; None = every track
PAD_KM = 120.0                         # map margin around the track swath

PROJ = ccrs.LambertConformal(central_longitude=-97.5, central_latitude=38.5, standard_parallels=(38.5,))
DATA_CRS = ccrs.PlateCarree()
STATES = cfeature.NaturalEarthFeature("cultural", "admin_1_states_provinces_lakes", "50m")
INK = "#222222"
mpl.rcParams.update({"font.size": 11, "text.color": INK, "axes.labelcolor": INK})
utc = lambda t: pd.Timestamp(t, unit="s")

In [ ]:
# metrics of all months
mfiles = sorted(glob.glob(DATA_DIR + "trackpcp_metrics_*.parquet"))
parts = []
for f in mfiles:
    period = re.search(r"metrics_(.+)\.parquet", os.path.basename(f)).group(1)      # START_END
    d = pd.read_parquet(f)
    d["period"] = period
    parts.append(d)
metrics = pd.concat(parts, ignore_index=True)

def parse_class_names(long_name):
    return {int(v): n.strip() for v, n in re.findall(r"(-?\d+)\s*=\s*([^,;]+)", str(long_name).split(":")[-1])}

class_names = {}
for period in metrics["period"].unique():
    with xr.open_dataset(f"{STATS_DIR}{STATS_FILEBASE}{period}.nc", decode_times=False, mask_and_scale=False) as ds:
        class_names[period] = parse_class_names(ds["track_class"].attrs.get("long_name", ""))
metrics["class_name"] = [class_names[p].get(int(c), f"class_{int(c)}") for p, c in zip(metrics["period"], metrics["class_value"])]
cand = metrics if CLASS_VALUES is None else metrics[metrics["class_value"].isin(CLASS_VALUES)]
top = cand.nlargest(NTOP, RANK_BY).reset_index(drop=True)
print(len(metrics), "tracks in", len(mfiles), "months; ranking by", RANK_BY)

## Table view: the highest-ranked tracks

In [ ]:
cols = ["period", "track_number", "class_name", "start_time", "duration_h", "accum_max_mm", "accum_vol_1e6m3",
        "local_depth_mm_15km", "local_depth_mm_45km", "local_depth_mm_99km", "local_rate_mm_h_45km",
        "area_ge50mm_km2", "cluster_area_km2", "start_lat", "start_lon"]
cols = [c for c in cols if c in cand.columns]
t = cand.nlargest(15, RANK_BY)[cols].copy()
t["start_time"] = pd.to_datetime(t["start_time"], unit="s").dt.strftime("%Y-%m-%d %H:%M")
t["start_lon"] = np.where(t["start_lon"] > 180, t["start_lon"] - 360, t["start_lon"])
t.round(1).reset_index(drop=True)

In [ ]:
# grid, pixel table access, pixel files
grid = xr.open_dataset(DATA_DIR + "trackpcp_grid.nc")
lon2d = grid["longitude"].values
lon2d = np.where(lon2d > 180, lon2d - 360, lon2d)
lat2d = grid["latitude"].values
NY, NX = lat2d.shape
xy_all = PROJ.transform_points(DATA_CRS, lon2d, lat2d)[..., :2]

def track_pixels(period, track):
    f = f"{DATA_DIR}trackpcp_pixels_{period}.parquet"
    return pq.read_table(f, filters=[("track_number", "==", int(track))]).to_pandas()

def file_time(f):
    n = os.path.basename(f)
    return pd.Timestamp(f"{n[9:13]}-{n[13:15]}-{n[15:17]} {n[18:20]}:{n[20:22]}:{n[22:24]}")

_pixel_times = {}
def pixel_file_at(period, t):
    if period not in _pixel_times:
        names = sorted(n for n in os.listdir(f"{PIXEL_ROOT}{period}/") if n.startswith("mcstrack_"))
        times = np.array([pd.Timestamp(f"{n[9:13]}-{n[13:15]}-{n[15:17]} {n[18:20]}:{n[20:22]}:{n[22:24]}").timestamp() for n in names])
        _pixel_times[period] = (names, times)
    names, times = _pixel_times[period]
    return f"{PIXEL_ROOT}{period}/{names[int(np.argmin(np.abs(times - t)))]}"

def key_times(row):
    # start, peak area, peak hourly rain volume (step 1 table if present), end
    period, k = row["period"], int(row["track_number"])
    with xr.open_dataset(f"{STATS_DIR}{STATS_FILEBASE}{period}.nc", decode_times=False, mask_and_scale=False) as ds:
        bt = ds["base_time"].values[k - 1]
        area = ds["total_area"].values[k - 1] if "total_area" in ds else ds["area"].values[k - 1]
    ok = np.isfinite(bt)
    t_peak_area = bt[ok][int(np.nanargmax(np.where(np.isfinite(area[ok]), area[ok], -1)))]
    t_peak_rain = None
    f1 = f"{DATA_DIR}mcs_tracks_final_qpe_{period}.parquet"
    if os.path.isfile(f1):
        d = pd.read_parquet(f1, columns=["track_number", "base_time", "total_rainvolume"])
        d = d[d["track_number"] == k]
        if len(d) and d["total_rainvolume"].notna().any():
            t_peak_rain = float(d.loc[d["total_rainvolume"].idxmax(), "base_time"])
    if t_peak_rain is None:
        t_peak_rain = 0.5 * (row["start_time"] + row["end_time"])
    return [("start", row["start_time"]), ("peak area", t_peak_area), ("peak hourly rain", t_peak_rain), ("end", row["end_time"])]

In [ ]:
LEVELS = [1, 10, 20, 30, 40, 50, 75, 100, 150, 200, 250]       # mm, accumulation scale shared by all tracks
ACC_CMAP = cmaps.WhBlGrYeRe.resampled(len(LEVELS) + 1)
ACC_NORM = mpl.colors.BoundaryNorm(LEVELS, ACC_CMAP.N, extend="both")
DBZ_CMAP, DBZ_NORM = mpl.colormaps["gist_ncar"], mpl.colors.Normalize(-10, 70)

def style_ax(ax, extent):
    ax.set_extent(extent, crs=PROJ)
    ax.add_feature(STATES, facecolor="none", edgecolor="#8c8c8c", linewidth=0.5, zorder=4)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), edgecolor="#555555", linewidth=0.6, zorder=4)
    ax.spines["geo"].set_edgecolor("#bbbbbb")

def plot_track(row, rank):
    period, k = row["period"], int(row["track_number"])
    px = track_pixels(period, k)
    iy, ix = np.divmod(px["pix_index"].values.astype(np.int64), NX)
    xs, ys = xy_all[iy, ix, 0], xy_all[iy, ix, 1]
    cx, cy = 0.5 * (xs.min() + xs.max()), 0.5 * (ys.min() + ys.max())
    half = max(xs.max() - xs.min(), ys.max() - ys.min()) / 2 + PAD_KM * 1000.0
    extent = [cx - half, cx + half, cy - half, cy + half]
    amap = np.full(NY * NX, np.nan, dtype=np.float32)
    amap[px["pix_index"].values] = px["accum_mm"].values
    amap = amap.reshape(NY, NX)
    times = key_times(row)

    fig = plt.figure(figsize=(21, 5.6), dpi=100)
    gs = gridspec.GridSpec(2, 5, figure=fig, height_ratios=[1, 0.045], hspace=0.12, wspace=0.04)
    ax = fig.add_subplot(gs[0, 0], projection=PROJ)
    style_ax(ax, extent)
    pm = ax.pcolormesh(lon2d, lat2d, np.ma.masked_invalid(amap), transform=DATA_CRS, cmap=ACC_CMAP, norm=ACC_NORM, shading="auto", zorder=2)
    ax.set_title(f"Accumulated precipitation (max {row['accum_max_mm']:.0f} mm)", fontsize=11, color=INK)
    cax = fig.add_subplot(gs[1, 0])
    cb = fig.colorbar(pm, cax=cax, orientation="horizontal", ticks=[1, 10, 30, 50, 100, 200])
    cb.set_label("Accumulated precipitation over the track lifetime (mm)", color=INK)
    for j, (name, t) in enumerate(times, start=1):
        f = pixel_file_at(period, t)
        with xr.open_dataset(f, decode_times=False, mask_and_scale=False) as ds:
            dbz = np.asarray(ds["dbz_comp"].values).squeeze().astype(float)
            ctn = np.asarray(ds["cloudtracknumber"].values).squeeze()
        a = fig.add_subplot(gs[0, j], projection=PROJ)
        style_ax(a, extent)
        pmz = a.pcolormesh(lon2d, lat2d, np.ma.masked_where(~(dbz >= 10), dbz), transform=DATA_CRS, cmap=DBZ_CMAP, norm=DBZ_NORM,
                           shading="auto", zorder=2, alpha=0.9)
        if (ctn == k).any():
            a.contour(lon2d, lat2d, (ctn == k).astype(float), levels=[0.5], colors="k", linewidths=1.1, transform=DATA_CRS, zorder=5)
        a.set_title(f"{name}: {file_time(f):%m-%d %H:%M} UTC", fontsize=11, color=INK)
    cax = fig.add_subplot(gs[1, 1:])
    cb = fig.colorbar(pmz, cax=cax, orientation="horizontal")
    cb.set_label("Composite reflectivity (dBZ); black line: the track's mask", color=INK)
    fig.suptitle(f"#{rank}  track {k} ({row['class_name']}), {utc(row['start_time']):%Y-%m-%d %H:%M} - {utc(row['end_time']):%Y-%m-%d %H:%M} UTC, "
                 f"{row['duration_h']:.1f} h   |   {RANK_BY} = {row[RANK_BY]:.0f} mm, volume {row['accum_vol_1e6m3']:.0f} x 10^6 m^3, "
                 f"area >= 50 mm {row.get('area_ge50mm_km2', float('nan')):.0f} km^2",
                 y=1.03, fontsize=13, fontweight="bold", color=INK)
    fig.savefig(f"{FIG_DIR}top_track_{rank}_{period[:8]}_{k}.png", dpi=130, bbox_inches="tight", facecolor="white")
    return fig

## The top tracks

In [ ]:
for i, row in top.iterrows():
    plot_track(row, i + 1)
    plt.show()